# Notebook 5: Model Evaluation

**Purpose:** This notebook evaluates the best-saved AtrionNet model on the held-out test set.
The test set was never seen during training or validation.

This notebook computes and reports:
- Instance-level Precision, Recall, and F1-Score
- Mean Average Precision (mAP @ IoU=0.5)
- Per-record breakdown of True Positives, False Positives, and False Negatives

**These are the final, publishable numbers for your thesis.**

## Step 1: Setup

In [ ]:
import sys
import os
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import DataLoader

PROJECT_ROOT = str(Path(os.getcwd()).parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## Step 2: Load the Best Saved Model

We load the weights that were saved whenever the validation F1 improved during training.

In [ ]:
from src.modeling.atrion_net import AtrionNetHybrid

MODEL_PATH = os.path.join(PROJECT_ROOT, 'weights', 'atrion_hybrid_best.pth')

model = AtrionNetHybrid(in_channels=12).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print(f"Model loaded from: {MODEL_PATH}")

## Step 3: Load the Held-out Test Set

The test indices were saved during training. We use the exact same held-out records.

In [ ]:
from src.data_pipeline.ludb_loader import LUDBLoader
from src.data_pipeline.instance_dataset import AtrionInstanceDataset

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw', 'ludb')
TEST_IDX_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'test_indices.npy')

loader = LUDBLoader(DATA_DIR)
signals, annotations = loader.get_all_data()

idx_test = np.load(TEST_IDX_PATH)
test_ds  = AtrionInstanceDataset(signals[idx_test], [annotations[i] for i in idx_test], is_train=False)
test_loader = DataLoader(test_ds, batch_size=1)  # One record at a time for per-record metrics

print(f"Test set size: {len(test_ds)} records")

## Step 4: Run Evaluation on All Test Records

For each test record we:
1. Run the model to get heatmap and width predictions
2. Extract detected P-wave instances using `find_peaks`
3. Match each prediction to a ground truth using 1D-IoU
4. Count True Positives, False Positives, and False Negatives

In [ ]:
from src.engine.atrion_evaluator import compute_instance_metrics, calculate_mAP

all_tp_lists = []
all_scores   = []
total_gt     = 0
record_results = []

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        sig  = batch['signal'].to(DEVICE)
        out  = model(sig)

        actual_idx = idx_test[i]
        targets = [{'span': (o, f)} for o, p, f in annotations[actual_idx]['p_waves']]

        res = compute_instance_metrics(
            out['heatmap'][0:1].cpu().numpy(),
            out['width'][0:1].cpu().numpy(),
            targets
        )

        all_tp_lists.append(res['tp_list'])
        all_scores.append(res['scores'])
        total_gt += res['n_gt']
        record_results.append(res)

print(f"Evaluated {len(record_results)} test records.")

## Step 5: Compute and Display Final Metrics

These are the final, thesis-quality results computed on the strictly held-out test set.

In [ ]:
m_ap, rec_curve, pre_curve = calculate_mAP(all_tp_lists, all_scores, total_gt)

avg_precision = float(np.mean([r['precision'] for r in record_results]))
avg_recall    = float(np.mean([r['recall']    for r in record_results]))
avg_f1        = float(np.mean([r['f1']        for r in record_results]))

total_tp = sum(r['tp'] for r in record_results)
total_fp = sum(r['fp'] for r in record_results)
total_fn = sum(r['fn'] for r in record_results)

print("=====================================================")
print(" ATRION-NET v4.0 — TEST SET RESULTS (Real LUDB Data)")
print("=====================================================")
print(f"  Precision : {avg_precision:.4f}")
print(f"  Recall    : {avg_recall:.4f}")
print(f"  F1 Score  : {avg_f1:.4f}")
print(f"  mAP@0.5   : {m_ap:.4f}")
print("-----------------------------------------------------")
print(f"  True Positives  (TP): {total_tp}")
print(f"  False Positives (FP): {total_fp}")
print(f"  False Negatives (FN): {total_fn}")
print(f"  Total Ground Truth  : {total_gt}")
print("=====================================================")

## Step 6: Per-Record Result Table

This table shows how the model performed on each individual test record.

In [ ]:
print(f"{'Record':>8} | {'GT':>4} | {'TP':>4} | {'FP':>4} | {'FN':>4} | {'Precision':>10} | {'Recall':>8} | {'F1':>8}")
print("-" * 70)
for i, res in enumerate(record_results):
    n_gt = res['n_gt']
    tp   = res['tp']
    fp   = res['fp']
    fn   = res['fn']
    prec = res['precision']
    rec  = res['recall']
    f1   = res['f1']
    print(f"    {i+1:>4} | {n_gt:>4} | {tp:>4} | {fp:>4} | {fn:>4} | {prec:>10.3f} | {rec:>8.3f} | {f1:>8.3f}")

---
**Evaluation complete. Proceed to `06_benchmarking/benchmarking.ipynb` for the ablation study.**